In [1]:
from conda_forge_tick.lazy_json_backends import LazyJson, lazy_json_override_backends

In [9]:
!rm -rf node_attrs pr_info pr_json version_pr_info

In [12]:
with lazy_json_override_backends(["github"], use_file_cache=True):
    attrs = LazyJson("node_attrs/ngmix.json")
    print(len(attrs.data))

28


In [8]:
from collections.abc import Collection, Mapping


def _sync_node(data, seen=None):
    seen = seen or []

    if isinstance(data, LazyJson):
        data.data

    if isinstance(data, Mapping):
        for v in data.values():
            if v not in seen:
                seen.append(v)
                seen = _sync_node(v, seen=seen)
    elif (
        isinstance(data, Collection)
        and not isinstance(data, str)
        and not isinstance(data, bytes)
    ):
        for v in data:
            if v not in seen:
                seen.append(v)
                seen = _sync_node(v, seen=seen)

    return seen


with lazy_json_override_backends(["github"]):
    ngmix = LazyJson("node_attrs/ngmix.json")
    _sync_node(ngmix)
    ngmix2 = LazyJson("node_attrs/ngmix.json")
    _sync_node(ngmix2)

    print(ngmix == ngmix2)

    with ngmix["pr_info"] as pri:
        pri.clear()
    print(ngmix == ngmix2)

    del ngmix.data["pr_info"]
    print(ngmix == ngmix2)

True
False
False


In [5]:
import hashlib


def _get_names_for_job(names, job, n_jobs):
    job_index = job - 1
    return [
        node_id
        for node_id in names
        if abs(int(hashlib.sha1(node_id.encode("utf-8")).hexdigest(), 16)) % n_jobs
        == job_index
    ]


print(_get_names_for_job(["devtools"], 3, 3))

['devtools']


In [10]:
s = "\tblah"

In [12]:
print(s)

	blah


In [13]:
s.startswith("\t")

True

In [1]:
import secrets
import time

RNG = secrets.SystemRandom()


def _retry_sequence(num_tries=20, base=2, factor=0.01):
    for i in range(num_tries):
        start = factor * base**i
        end = factor * base ** (i + 1)
        time.sleep(RNG.uniform(start, end))
        yield i


for tr in _retry_sequence():
    print(tr)

0
1
2
3
4
5
6
7
8
9
10
11


KeyboardInterrupt: 